In [2]:
import sys
from pathlib import Path 
import pandas as pd 
import numpy as np 

project_root = Path.cwd().parent
sys.path.append(str(project_root))
print(project_root)

c:\Users\Sahiti Putcha\Documents\Equity-Risk-Portfolio-Analysis


In [3]:
daily_returns = pd.read_csv(
    project_root / "data" / "processed" / "daily_returns.csv",
    index_col =0,
    parse_dates = True
)
daily_returns.head()

,GLD,TLT,SPY,JPM,XOM,AAPL,JNJ,KO
Date,,,,,,,,
2016-01-05,0.002819,-0.004034,0.001691,0.001729,0.008521,-0.025059,0.004180,0.003538
2016-01-06,0.014441,0.013475,-0.012614,-0.014436,-0.008321,-0.019570,-0.005055,-0.005405
2016-01-07,0.014140,0.001795,-0.023992,-0.040440,-0.016006,-0.042204,-0.011654,-0.016541
2016-01-08,-0.004428,0.004479,-0.010977,-0.022399,-0.020202,0.005288,-0.010683,-0.002643
2016-01-11,-0.008895,-0.010944,0.000990,-0.001528,-0.013388,0.016192,-0.006010,0.001687


Annual Returns

In [4]:
annual_returns = (1+ daily_returns).prod()**(252/len(daily_returns)) - 1
annual_returns

GLD     0.131987
TLT    -0.007847
SPY     0.151494
JPM     0.206839
XOM     0.114620
AAPL    0.282896
JNJ     0.123151
KO      0.098636
dtype: float64

Annual Volatility

In [5]:
annual_volatility = daily_returns.std()*np.sqrt(252)
annual_volatility

GLD     0.162449
TLT     0.147469
SPY     0.178086
JPM     0.274041
XOM     0.279524
AAPL    0.289163
JNJ     0.184832
KO      0.181695
dtype: float64

Sharpe Ratio

In [6]:
sharpe_ratio = annual_returns/annual_volatility
sharpe_ratio

GLD     0.812481
TLT    -0.053212
SPY     0.850677
JPM     0.754775
XOM     0.410056
AAPL    0.978327
JNJ     0.666286
KO      0.542866
dtype: float64

Corelation Matrix

In [7]:
corelation_matrix = daily_returns.corr()
corelation_matrix

,GLD,TLT,SPY,JPM,XOM,AAPL,JNJ,KO
GLD,1.000000,0.259907,0.076342,-0.063801,0.036421,0.035133,0.043699,0.044607
TLT,0.259907,1.000000,-0.155990,-0.307057,-0.237250,-0.092116,-0.086755,-0.071796
SPY,0.076342,-0.155990,1.000000,0.701697,0.486361,0.741368,0.428062,0.491976
JPM,-0.063801,-0.307057,0.701697,1.000000,0.503279,0.407726,0.329744,0.385279
XOM,0.036421,-0.237250,0.486361,0.503279,1.000000,0.277226,0.283460,0.355109
AAPL,0.035133,-0.092116,0.741368,0.407726,0.277226,1.000000,0.289435,0.323400
JNJ,0.043699,-0.086755,0.428062,0.329744,0.283460,0.289435,1.000000,0.494959
KO,0.044607,-0.071796,0.491976,0.385279,0.355109,0.323400,0.494959,1.000000


Portfolio Returns

In [8]:
weights = np.array([1/8]*8)
portfolio_returns = daily_returns.dot(weights)
portfolio_returns.head()

Date
2016-01-05   -0.000827
2016-01-06   -0.004686
2016-01-07   -0.016863
2016-01-08   -0.007696
2016-01-11   -0.002737
dtype: float64

Portfolio Annual Return

In [9]:
portfolio_annual_returns = (1+portfolio_returns).prod()**(252/len(portfolio_returns)) - 1
print(f"portfolio annual returns: {portfolio_annual_returns:.2%}")

portfolio annual returns: 15.34%


Portfolio Annual Volatility

In [10]:
portfolio_annual_volatility = (1+portfolio_returns).std()*np.sqrt(252)
print(f"portfolio annual voalility: {portfolio_annual_volatility:.2%}")

portfolio annual voalility: 12.58%


Portfolio Sharpe Ratio

In [11]:
portfolio_sharpe = portfolio_annual_returns/portfolio_annual_volatility
print(f"portfoilio sharpe: {portfolio_sharpe:.3f}")

portfoilio sharpe: 1.219


Portfolio Value

In [12]:
portfolio_value = (1 + portfolio_returns).cumprod()
portfolio_value.head()

Date
2016-01-05    0.999173
2016-01-06    0.994491
2016-01-07    0.977721
2016-01-08    0.970197
2016-01-11    0.967542
dtype: float64

Running Maximum

In [13]:
running_max = portfolio_value.cummax()
running_max.head()

Date
2016-01-05    0.999173
2016-01-06    0.999173
2016-01-07    0.999173
2016-01-08    0.999173
2016-01-11    0.999173
dtype: float64

Drawdown

In [14]:
drawdown = (portfolio_value - running_max) / running_max
drawdown.head()

Date
2016-01-05    0.000000
2016-01-06   -0.004686
2016-01-07   -0.021469
2016-01-08   -0.029000
2016-01-11   -0.031658
dtype: float64

Maximum Drawdown

In [15]:
max_drawdown = drawdown.min()
print(f"maximum drawdown :{max_drawdown:.2%}")

maximum drawdown :-26.93%


Value at Risk

In [16]:
var_95 = portfolio_returns.quantile(0.05)
print(f"var 95: {var_95:.2%}")

var 95: -1.09%


Load Market Returns(SPY)

In [17]:
market_returns = daily_returns["SPY"]
market_returns.head()

Date
2016-01-05    0.001691
2016-01-06   -0.012614
2016-01-07   -0.023992
2016-01-08   -0.010977
2016-01-11    0.000990
Name: SPY, dtype: float64

In [18]:
type(market_returns)

pandas.core.series.Series

Market Variance

In [19]:
market_variance = market_returns.var()
print("market variance: ",market_variance)

market variance:  0.00012585176791096706


Beta for each asset

In [20]:
beta = {}
for asset in daily_returns.columns:
    covariance = daily_returns[asset].cov(market_returns)
    beta[asset] = covariance/market_variance
beta = pd.Series(beta)
beta

GLD     0.069639
TLT    -0.129171
SPY     1.000000
JPM     1.079777
XOM     0.763391
AAPL    1.203777
JNJ     0.444277
KO      0.501946
dtype: float64

Asset Summary Table

In [21]:
asset_summary = pd.DataFrame({
    "annual return": annual_returns,
    "annual volatility": annual_volatility,
    "sharpe ratio": sharpe_ratio,
    "beta" : beta,

})
asset_summary

,annual return,annual volatility,sharpe ratio,beta
GLD,0.131987,0.162449,0.812481,0.069639
TLT,-0.007847,0.147469,-0.053212,-0.129171
SPY,0.151494,0.178086,0.850677,1.000000
JPM,0.206839,0.274041,0.754775,1.079777
XOM,0.114620,0.279524,0.410056,0.763391
AAPL,0.282896,0.289163,0.978327,1.203777
JNJ,0.123151,0.184832,0.666286,0.444277
KO,0.098636,0.181695,0.542866,0.501946


Save Asset Summary

In [22]:
asset_summary.index.name= "asset"
asset_summary.to_csv(
    project_root / "data" / "processed" / "assrt_summary.csv"
    
)

print("asset_summary.csv saved!")

asset_summary.csv saved!


Export Corelation Matrix

In [23]:
corelation_matrix.to_csv(
    project_root / "data" / "processed" / "corelaion_matrix.csv"
)

print("corelation_matrix.csv saved!")

corelation_matrix.csv saved!


Create Portfolio History 

In [24]:
portfolio_history = pd.DataFrame({
    "portfolio value" : portfolio_value,
    "running max" : running_max,
    "drawdown" : drawdown,

})
portfolio_history.head()

,portfolio value,running max,drawdown
Date,,,
2016-01-05,0.999173,0.999173,0.000000
2016-01-06,0.994491,0.999173,-0.004686
2016-01-07,0.977721,0.999173,-0.021469
2016-01-08,0.970197,0.999173,-0.029000
2016-01-11,0.967542,0.999173,-0.031658


Export Portfolio History

In [25]:
portfolio_history.to_csv(
    project_root / "data" /"processed"/"portfolio_history.csv"
)
print("portfolio_history.csv saved!")

portfolio_history.csv saved!


Create Portfolio Summary 

In [26]:
portfolio_summary = pd.DataFrame({
    "annual return": [portfolio_annual_returns],
    "annual volatility" : [portfolio_annual_volatility],
    "portfolio annual volatility":[portfolio_annual_volatility],
    "sharpe ratio" : [portfolio_sharpe],
    "maximum drawdown" : [max_drawdown],
    "95% var" : [var_95]

})
portfolio_summary

,annual return,annual volatility,portfolio annual volatility,sharpe ratio,maximum drawdown,95% var
0,0.15338,0.125813,0.125813,1.219116,-0.26927,-0.010934


Export Portfolio Summary

In [27]:
portfolio_summary.to_csv(
    project_root / "data" / "processed" / "portfolio_summary.csv" ,
    index = False
)
print("portfolio_summary.csv saved!")

portfolio_summary.csv saved!


PowerBI Corelation Heatmap

In [28]:
corelation_long = (
    corelation_matrix 
    .reset_index()
    .rename(columns = {"index":"asset1"})
    .melt(
        id_vars = "asset1",
        var_name = "asset2",
        value_name = "corelation"
    )
)
corelation_long.head()

,asset1,asset2,corelation
0,GLD,GLD,1.000000
1,TLT,GLD,0.259907
2,SPY,GLD,0.076342
3,JPM,GLD,-0.063801
4,XOM,GLD,0.036421


In [29]:
corelation_long.to_csv(
    project_root / "data" / "processed" / "corelation_long.csv",
    index = False
)
print("saved!")

saved!
